In [ ]:
import os
import torch
import torch.nn as nn
from google.colab import drive
drive.mount('/content/drive')

from transformers import (
    AutoTokenizer,
    AutoModel
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
'''

import torch

obj = torch.load(
    '/content/drive/MyDrive/Weights/model_weights_binary.pth',
    map_location='cpu'
)

print(type(obj))
'''

"\n\nimport torch\n\nobj = torch.load(\n    '/content/drive/MyDrive/Weights/model_weights_binary.pth',\n    map_location='cpu'\n)\n\nprint(type(obj))\n"

In [ ]:
import os

print(os.path.getsize('/content/drive/MyDrive/Weights/model_weights_binary.pth'))
print(os.path.getsize('/content/drive/MyDrive/Weights/model_weights_multiclass.pth'))

498639548
498759020


In [ ]:
# Configuración

model_name = "microsoft/codebert-base"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [ ]:
# Modo híbrido

class VulnerabilityModel(nn.Module):

    def __init__(self, codebert):

        super(VulnerabilityModel, self).__init__()

        self.codebert = codebert

        # Clasificador binario
        self.classifier_bin = nn.Linear(768, 2)

        # Clasificador multiclase
        self.classifier_mul = nn.Linear(768, 29)

    def forward(self, input_ids, attention_mask):

        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Token CLS
        latent_vector = outputs.last_hidden_state[:, 0, :]

        logits_bin = self.classifier_bin(latent_vector)

        logits_mul = self.classifier_mul(latent_vector)

        return logits_bin, logits_mul

In [ ]:
# Inicializar tokenizer y modelo

print("Cargando CodeBERT...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModel.from_pretrained(model_name)

model = VulnerabilityModel(base_model).to(device)

print("Modelo híbrido creado")

Cargando CodeBERT...


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo híbrido creado


In [ ]:
# Cargar pesos entrenados

print("Cargando pesos entrenados...")

# MODELO BINARIO

checkpoint_bin = torch.load(
    '/content/drive/MyDrive/Weights/model_weights_binary.pth',
    map_location=device
)

# MODELO MULTICLASE

checkpoint_mul = torch.load(
    '/content/drive/MyDrive/Weights/model_weights_multiclass.pth',
    map_location=device
)

# COPIAR PESOS BINARIOS

with torch.no_grad():

    model.classifier_bin.weight.copy_(
        checkpoint_bin['classifier.weight']
    )

    model.classifier_bin.bias.copy_(
        checkpoint_bin['classifier.bias']
    )
# COPIAR PESOS MULTICLASE

with torch.no_grad():

    model.classifier_mul.weight.copy_(
        checkpoint_mul['classifier.weight']
    )

    model.classifier_mul.bias.copy_(
        checkpoint_mul['classifier.bias']
    )

print("Pesos cargados correctamente")

Cargando pesos entrenados...


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Pesos cargados correctamente


In [ ]:
print(checkpoint_bin.keys())
print(checkpoint_mul.keys())

odict_keys(['codebert.embeddings.word_embeddings.weight', 'codebert.embeddings.token_type_embeddings.weight', 'codebert.embeddings.LayerNorm.weight', 'codebert.embeddings.LayerNorm.bias', 'codebert.embeddings.position_embeddings.weight', 'codebert.encoder.layer.0.attention.self.query.weight', 'codebert.encoder.layer.0.attention.self.query.bias', 'codebert.encoder.layer.0.attention.self.key.weight', 'codebert.encoder.layer.0.attention.self.key.bias', 'codebert.encoder.layer.0.attention.self.value.weight', 'codebert.encoder.layer.0.attention.self.value.bias', 'codebert.encoder.layer.0.attention.output.dense.weight', 'codebert.encoder.layer.0.attention.output.dense.bias', 'codebert.encoder.layer.0.attention.output.LayerNorm.weight', 'codebert.encoder.layer.0.attention.output.LayerNorm.bias', 'codebert.encoder.layer.0.intermediate.dense.weight', 'codebert.encoder.layer.0.intermediate.dense.bias', 'codebert.encoder.layer.0.output.dense.weight', 'codebert.encoder.layer.0.output.dense.bias', 

In [ ]:
# Inferecia manual

def predecir_codigo(texto):

    model.eval()

    encoding = tokenizer(
        texto,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)

    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():

        logits_bin, logits_mul = model(
            input_ids,
            attention_mask
        )

        pred_bin = torch.argmax(logits_bin, dim=1).item()

        pred_mul = torch.argmax(logits_mul, dim=1).item()

    clases_binarias = {
        0: "Seguro",
        1: "Vulnerable"
    }

    print("=" * 50)
    print("RESULTADOS")
    print("=" * 50)

    print(f"Clasificación Binaria: {clases_binarias[pred_bin]}")

    print(f"Clase Multivulnerabilidad: {pred_mul}")

In [ ]:
# ==========================================
# PRUEBA MANUAL DE CÓDIGO
# ==========================================

def predecir_codigo(texto):

    model.eval()

    encoding = tokenizer(
        texto,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)

    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():

        # ==============================
        # PREDICCIONES
        # ==============================

        logits_bin, logits_mul = model(
            input_ids,
            attention_mask
        )

        # ==============================
        # BINARIO
        # ==============================

        probs_bin = torch.softmax(logits_bin, dim=1)

        # Probabilidades
        prob_seguro = probs_bin[0][0].item()
        prob_vulnerable = probs_bin[0][1].item()

        # Umbral personalizado
        threshold = 0.45

        if prob_vulnerable > threshold:
            pred_bin = 1
            confianza_bin = prob_vulnerable
        else:
            pred_bin = 0
            confianza_bin = prob_seguro

        # ==============================
        # MULTICLASE
        # ==============================

        probs_mul = torch.softmax(logits_mul, dim=1)

        pred_mul = torch.argmax(
            probs_mul,
            dim=1
        ).item()

        confianza_mul = probs_mul[0][pred_mul].item()

    # ======================================
    # CLASES BINARIAS
    # ======================================

    clases_binarias = {
        0: "Seguro",
        1: "Vulnerable"
    }

    # ======================================
    # CLASES MULTIVULNERABILIDAD
    # ======================================

    labels = [
        'PenetrationTestingScripts',
        'cybersecurity-penetration-testing',
        'owtf',
        'Python-Penetration-Testing-for-Developers',
        'Hands-On-Penetration-Testing-with-Python',
        'thieves-tools',
        'Python-Penetration-Testing-Cookbook',
        'Penetration_Testing',
        'Python-for-Offensive-PenTest',
        'Penetration-Testing-with-Shellcode',
        'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-Second-Edition',
        'PenTestScripts',
        'Tricks-Web-Penetration-Tester',
        'Penetration-Testing-Study-Notes',
        'Broken-Droid-Factory',
        'Ethical-Hacking-Scripts',
        'Effective-Python-Penetration-Testing',
        'Mastering-Machine-Learning-for-Penetration-Testing',
        'PenTesting',
        'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-4E',
        'hackipy',
        'AggressorAssessor',
        'Hands-On-AWS-Penetration-Testing-with-Kali-Linux',
        'GWT-Penetration-Testing-Toolset',
        'diff-droid',
        'SNAP_R',
        'Advanced-Infrastructure-Penetration-Testing',
        'Hands-On-Bug-Hunting-for-Penetration-Testers',
        'Nojle'
    ]

    descripciones = {
    'PenetrationTestingScripts':
        'Scripts utilizados para pruebas de penetración y evaluación de seguridad.',

    'cybersecurity-penetration-testing':
        'Herramientas y técnicas ofensivas utilizadas en ciberseguridad.',

    'owtf':
        'Framework para pruebas automatizadas de seguridad en aplicaciones web.',

    'Python-Penetration-Testing-for-Developers':
        'Código relacionado con pruebas ofensivas desarrolladas en Python.',

    'Hands-On-Penetration-Testing-with-Python':
        'Ejemplos prácticos de pruebas de penetración utilizando Python.',

    'thieves-tools':
        'Conjunto de herramientas potencialmente utilizadas para explotación ofensiva.',

    'Python-Penetration-Testing-Cookbook':
        'Colección de técnicas y recetas para pentesting en Python.',

    'Penetration_Testing':
        'Scripts generales para pruebas de penetración.',

    'Python-for-Offensive-PenTest':
        'Automatización ofensiva y explotación utilizando Python.',

    'Penetration-Testing-with-Shellcode':
        'Código relacionado con shellcodes y explotación de memoria.',

    'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-Second-Edition':
        'Técnicas avanzadas de pentesting utilizando Kali Linux.',

    'PenTestScripts':
        'Scripts automatizados para evaluación ofensiva.',

    'Tricks-Web-Penetration-Tester':
        'Técnicas comunes de explotación web.',

    'Penetration-Testing-Study-Notes':
        'Notas y ejemplos educativos relacionados con pentesting.',

    'Broken-Droid-Factory':
        'Herramientas de análisis y explotación para Android.',

    'Ethical-Hacking-Scripts':
        'Scripts asociados con hacking ético y auditorías de seguridad.',

    'Effective-Python-Penetration-Testing':
        'Implementaciones ofensivas eficientes en Python.',

    'Mastering-Machine-Learning-for-Penetration-Testing':
        'Uso de machine learning aplicado a ciberseguridad ofensiva.',

    'PenTesting':
        'Código general de pruebas de penetración.',

    'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-4E':
        'Pentesting avanzado con Kali Linux.',

    'hackipy':
        'Herramientas ofensivas desarrolladas en Python.',

    'AggressorAssessor':
        'Scripts relacionados con automatización ofensiva.',

    'Hands-On-AWS-Penetration-Testing-with-Kali-Linux':
        'Pruebas ofensivas dirigidas a entornos AWS.',

    'GWT-Penetration-Testing-Toolset':
        'Herramientas para pruebas de seguridad específicas.',

    'diff-droid':
        'Análisis diferencial y evaluación de aplicaciones Android.',

    'SNAP_R':
        'Herramientas ofensivas y automatización de seguridad.',

    'Advanced-Infrastructure-Penetration-Testing':
        'Evaluación ofensiva de infraestructura y redes.',

    'Hands-On-Bug-Hunting-for-Penetration-Testers':
        'Detección práctica de vulnerabilidades.',

    'Nojle':
        'Repositorio asociado a herramientas de seguridad ofensiva.'
}

    # ======================================
    # RESULTADOS
    # ======================================

    print("=" * 50)
    print("RESULTADO")
    print("=" * 50)

    print(f"Clasificación: {clases_binarias[pred_bin]}")

    print(f"Confianza Binaria: {confianza_bin:.4f}")

    # ======================================
    # SI ES VULNERABLE
    # ======================================

    if pred_bin == 1:

        print("\nTipo de vulnerabilidad detectada:")

        tipo_vulnerabilidad = labels[pred_mul]

        print(tipo_vulnerabilidad)

        print("\nDescripción:")

        print(descripciones[tipo_vulnerabilidad])

        print(f"Confianza Multiclase: {confianza_mul:.4f}")

    print("=" * 50)


# ==========================================
# EJEMPLO DE USO
# ==========================================

codigo = """
import os

os.system("rm -rf /")
"""

predecir_codigo(codigo)

RESULTADO
Clasificación: Vulnerable
Confianza Binaria: 0.4617

Tipo de vulnerabilidad detectada:
Penetration-Testing-with-Shellcode

Descripción:
Código relacionado con shellcodes y explotación de memoria.
Confianza Multiclase: 0.0529
